<a href="https://colab.research.google.com/github/and1-lol/CRCMeme/blob/main/Value_Transfers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- CELL 1: JAVASCRIPT/HTML FRONTEND ---
# Copy the output of this cell into a file named index.html

html_content = """
<!DOCTYPE html>
<html lang='en'>
<head>
    <meta charset='UTF-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1.0'>
    <title>Craftcoin V3.0</title>
    <style>
        body { font-family: sans-serif; background: #121212; color: #e0e0e0; display: flex; margin: 0; height: 100vh; }
        #menu { width: 200px; background: #1f1f1f; padding: 20px; border-right: 1px solid #333; }
        #content { flex: 1; padding: 40px; overflow-y: auto; }
        button { display: block; width: 100%; padding: 10px; margin: 10px 0; background: #333; color: white; border: 1px solid #444; cursor: pointer; }
        button:hover { background: #444; }
        input { width: 100%; padding: 8px; margin: 5px 0; background: #222; color: white; border: 1px solid #444; }
        .card { background: #1f1f1f; padding: 20px; border-radius: 8px; border: 1px solid #333; }
        #log { background: #000; padding: 10px; height: 200px; overflow-y: scroll; font-family: monospace; font-size: 12px; border: 1px solid #333; }
    </style>
</head>
<body>

<div id='menu'>
    <h3>Craftcoin</h3>
    <button onclick='showPage("scout")'>To Scout</button>
    <button onclick='showPage("transfer")'>Transfer</button>
    <button onclick='showPage("ledger")'>Ledger</button>
    <button onclick='showPage("balance")'>Check Balance</button>
</div>

<div id='content'>
    <div id='view'></div>
</div>

<script>
    // Storage logic
    if (!localStorage.getItem('addr')) localStorage.setItem('addr', JSON.stringify({ 'a': 100, 'b': 50, 'c': 10 }));
    if (!localStorage.getItem('ledger')) localStorage.setItem('ledger', 'System Initialized');

    function getBalances() { return JSON.parse(localStorage.getItem('addr')); }
    function setBalances(b) { localStorage.setItem('addr', JSON.stringify(b)); }
    function addLog(msg) {
        let time = new Date().toLocaleString();
        let current = localStorage.getItem('ledger');
        localStorage.setItem('ledger', current + `\\n[${time}] ${msg}`);
    }

    function showPage(page) {
        const view = document.getElementById('view');
        if (page === 'scout') {
            view.innerHTML = `<div class='card'><h2>Scout</h2><input type='number' id='snum' placeholder='Guess 1-5'><input type='text' id='saddr' placeholder='Address'><button onclick='doScout()'>Submit</button></div>`;
        } else if (page === 'transfer') {
            view.innerHTML = `<div class='card'><h2>Transfer</h2><input type='text' id='u' placeholder='From'><input type='text' id='t' placeholder='To'><input type='number' id='a' placeholder='Amount'><button onclick='doTransfer()'>Send</button></div>`;
        } else if (page === 'ledger') {
            view.innerHTML = `<h2>Ledger</h2><div id='log'>${localStorage.getItem('ledger').replace(/\\n/g, '<br>')}</div>`;
        } else if (page === 'balance') {
            view.innerHTML = `<div class='card'><h2>Check Balance</h2><input type='text' id='baddr' placeholder='Address'><button onclick='doBalance()'>Check</button><p id='bres'></p></div>`;
        }
    }

    function doScout() {
        let num = document.getElementById('snum').value; let addr = document.getElementById('saddr').value;
        let win = Math.floor(Math.random() * 5) + 1;
        if (parseInt(num) === win) {
            let b = getBalances(); if(b[addr]!==undefined) { b[addr]+=20; setBalances(b); addLog(`Scouted 20 for ${addr}`); alert('Won 20!'); }
        } else alert('Lost! Correct was ' + win);
    }

    function doTransfer() {
        let u = document.getElementById('u').value; let t = document.getElementById('t').value; let amt = parseFloat(document.getElementById('a').value);
        let b = getBalances();
        if (b[u] >= amt && amt > 0) {
            b[u] -= amt; b[t] += amt; setBalances(b); addLog(`${u} sent ${amt} to ${t}`); alert('Success!');
        } else alert('Insufficient funds.');
    }

    function doBalance() {
        let addr = document.getElementById('baddr').value; let b = getBalances();
        document.getElementById('bres').innerText = b[addr] !== undefined ? `Balance: ${b[addr]} CC` : 'Not Found';
    }
    showPage('scout');
</script>
</body>
</html>
"""

print("--- FRONTEND HTML CODE ---")
print(html_content)

--- FRONTEND HTML CODE ---

<!DOCTYPE html>
<html lang='en'>
<head>
    <meta charset='UTF-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1.0'>
    <title>Craftcoin V3.0</title>
    <style>
        body { font-family: sans-serif; background: #121212; color: #e0e0e0; display: flex; margin: 0; height: 100vh; }
        #menu { width: 200px; background: #1f1f1f; padding: 20px; border-right: 1px solid #333; }
        #content { flex: 1; padding: 40px; overflow-y: auto; }
        button { display: block; width: 100%; padding: 10px; margin: 10px 0; background: #333; color: white; border: 1px solid #444; cursor: pointer; }
        button:hover { background: #444; }
        input { width: 100%; padding: 8px; margin: 5px 0; background: #222; color: white; border: 1px solid #444; }
        .card { background: #1f1f1f; padding: 20px; border-radius: 8px; border: 1px solid #333; }
        #log { background: #000; padding: 10px; height: 200px; overflow-y: scroll; font-family

In [ ]:
# --- CELL 2: ORIGINAL PYTHON CRAFTCOIN BACKEND ---
import random, datetime, hashlib, os, json
from flask import Flask, request, jsonify

LEDGER_DIR = '/content/CraftcoinData'
os.makedirs(LEDGER_DIR, exist_ok=True)
ADDR_FILE = os.path.join(LEDGER_DIR, 'addr.json')
LOG_FILE = os.path.join(LEDGER_DIR, 'log.txt')

# --- OG LOGIC FUNCTIONS ---
def load_addr():
    if os.path.exists(ADDR_FILE):
        with open(ADDR_FILE, 'r') as f: return json.load(f)
    return {'a': 0, 'b': 0, 'c': 0}

def save_addr(data):
    with open(ADDR_FILE, 'w') as f: json.dump(data, f, indent=2)

def led(message):
    time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(LOG_FILE, 'a') as f: f.write(f'[{time}] {message}\n')

def ee(): # OG scouting number generator
    return random.randint(1,10000)

app = Flask(__name__)

@app.route('/api/balance/<address>', methods=['GET'])
def balance_api(address):
    addr_data = load_addr()
    bal = addr_data.get(address.lower())
    return jsonify({"status": "success", "balance": bal}) if bal is not None else (jsonify({"error": "Invalid Address"}), 404)

@app.route('/api/scout', methods=['POST'])
def scout_api():
    data = request.get_json()
    addr_data = load_addr()
    target_num = ee()
    user_num = int(data.get('number', 0))
    address = data.get('address', '').lower()

    if user_num == target_num:
        if address in addr_data:
            addr_data[address] += 20
            save_addr(addr_data)
            led(f'Scouted 20 Craftcoin for {address} by entering {user_num}')
            return jsonify({"status": "success", "message": "You won 20 Craftcoin!"})
    return jsonify({"status": "fail", "message": f"Incorrect. The number was {target_num}"}), 400

@app.route('/api/transfer', methods=['POST'])
def transfer_api():
    data = request.get_json()
    addr_data = load_addr()
    u, t, amt = data.get('from'), data.get('to'), float(data.get('amount', 0))

    if addr_data.get(u, 0) >= amt and amt > 0:
        addr_data[u] -= amt
        addr_data[t] += amt
        save_addr(addr_data)
        led(f'Transaction: {u} sent {amt} Craftcoin to {t}')
        return jsonify({"status": "success"})
    return jsonify({"status": "error", "message": "Insufficient funds or invalid transfer"}), 400

if __name__ == '__main__':
    print("Starting OG Craftcoin Backend API...")
    app.run(host='0.0.0.0', port=5000)

ValueError: mount failed